# Dataset 2 — Steel Industry Energy Consumption (UCI)
## Etapa B — Python / Pandas

Este notebook parte da amostra de 20% gerada na Etapa A (Orange Data Mining).
Se estiver rodando no Colab, faça upload do arquivo `amostra_20pct.csv` antes de executar.

### 1. Carregar a amostra e renomear colunas

In [9]:
import pandas as pd

df = pd.read_csv('/content/steel industry energy.csv')

df = df.rename(columns={
    'Usage_kWh': 'Consumo_kWh',
    'Lagging_Current_Reactive.Power_kVarh': 'Reativa_Atrasada_kVarh',
    'Leading_Current_Reactive_Power_kVarh': 'Reativa_Adiantada_kVarh',
    'Lagging_Current_Power_Factor': 'FP_Atrasado',
    'Leading_Current_Power_Factor': 'FP_Adiantado'
})

df.head()

,Consumo_kWh,Reativa_Atrasada_kVarh,Reativa_Adiantada_kVarh,FP_Atrasado,FP_Adiantado,WeekStatus,Day_of_week,Load_Type
0,2.88,3.82,0.0,60.20,100.00,Weekend,Sunday,Light_Load
1,60.77,48.02,0.0,78.46,100.00,Weekday,Thursday,Maximum_Load
2,120.42,59.65,0.0,89.61,100.00,Weekday,Friday,Maximum_Load
3,3.13,0.00,16.6,100.00,18.53,Weekend,Saturday,Light_Load
4,58.86,20.99,0.0,94.19,100.00,Weekday,Friday,Medium_Load


### 2. Inspeção inicial

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe()

### 3. Maior consumo registrado e limiar de 75%

In [ ]:
consumo_max = df['Consumo_kWh'].max()
limiar_consumo = 0.75 * consumo_max

print(f"Consumo máximo registrado: {consumo_max:.2f} kWh")
print(f"Limiar (75% do máximo): {limiar_consumo:.2f} kWh")

### 4. Registros acima do limiar: quantidade e percentual

In [ ]:
df_alto_consumo = df[df['Consumo_kWh'] > limiar_consumo].copy()

qtd = len(df_alto_consumo)
pct = 100 * qtd / len(df)

print(f"Quantidade de registros acima do limiar: {qtd}")
print(f"Percentual do total da amostra: {pct:.2f}%")

df_alto_consumo.head()

### 5. Quantos desses registros são `Maximum_Load`?

In [ ]:
contagem_load = df_alto_consumo['Load_Type'].value_counts()
print(contagem_load)

qtd_max_load = (df_alto_consumo['Load_Type'] == 'Maximum_Load').sum()
pct_max_load = 100 * qtd_max_load / qtd

print(f"\n{qtd_max_load} de {qtd} registros de alto consumo ({pct_max_load:.2f}%) são Maximum_Load")

### 6. Fator de potência — definindo um limite coerente para "valores baixos"

Usaremos `FP_Atrasado`. Observação importante: olhar o percentil no **dataset inteiro**
não é útil aqui, porque nos períodos de consumo elevado o FP já é naturalmente alto
(cargas grandes e constantes tendem a operar com FP melhor). Por isso, o limite de
"FP baixo" é definido **dentro do próprio grupo de alto consumo** (25º percentil local).

In [ ]:
print(df_alto_consumo['FP_Atrasado'].describe())

limite_fp = df_alto_consumo['FP_Atrasado'].quantile(0.25)
print(f"\nLimite de FP_Atrasado (25º percentil dentro do grupo de alto consumo): {limite_fp:.2f}")

### 7. Consumo elevado + FP baixo simultaneamente

In [ ]:
df_critico = df_alto_consumo[df_alto_consumo['FP_Atrasado'] < limite_fp].copy()

print(f"Registros críticos (consumo alto + FP relativamente baixo): {len(df_critico)}")
print(f"Percentual do grupo de alto consumo: {100*len(df_critico)/qtd:.2f}%")

print("\nDistribuição de Load_Type no subconjunto crítico:")
print(df_critico['Load_Type'].value_counts())

print("\nDistribuição de WeekStatus no subconjunto crítico:")
print(df_critico['WeekStatus'].value_counts())

df_critico[['Consumo_kWh', 'FP_Atrasado']].describe()

### 8. Por que esse conjunto merece mais atenção da equipe de energia?

Esses registros representam os momentos em que a planta está **próxima do pico de consumo**
e, ao mesmo tempo, operando com o **fator de potência relativamente pior** entre os picos.
Isso significa que, além de puxar mais energia da rede, o processo está fazendo isso de forma
menos eficiente (maior componente reativa por kWh útil). Na prática, isso costuma implicar:

- maior custo de demanda/penalidade por baixo fator de potência na conta de energia;
- maior perda por efeito Joule na instalação elétrica;
- um bom ponto de partida para investigar equipamentos específicos (motores subdimensionados,
  partidas de máquinas, ausência de correção reativa local).

Como a maioria desses registros ocorre em dias úteis e está concentrada em `Maximum_Load`,
é um alvo natural para ações de manutenção ou compensação de reativos.

### (Opcional) Exportar os subconjuntos gerados

In [ ]:
df_alto_consumo.to_csv('alto_consumo.csv', index=False)
df_critico.to_csv('consumo_critico.csv', index=False)

# No Colab, para baixar os arquivos:
# from google.colab import files
# files.download('alto_consumo.csv')
# files.download('consumo_critico.csv')